# Evaluation of Formulas from First Order Logic

In this notebook we show how formulas from *first order logic* can be evaluated in TypeScript.

## The Axioms of Group Theory

To have a nontrivial example of formulas, we use the formulas from 
[group theory](https://en.wikipedia.org/wiki/Group_theory).  
A [group](https://en.wikipedia.org/wiki/Group_(mathematics)) is defined as a triple 
$$ \langle G, \mathrm{e}, \circ \rangle $$
where 
- $G$ is a non-empty set,
- $\mathrm{e}$ is an element from $G$, and
- $\circ:G \times G \rightarrow G$ is a binary function on $G$.
- Furthermore, the following axioms have to be satisfied:
  * $\forall x: \mathrm{e} \circ x = x$,
  * $\forall x: \exists{y}: y \circ x = \mathrm{e}$,
  * $\forall x: \forall y: \forall z: (x \circ y) \circ z = x \circ (y \circ z)$.
- A group is <em style="color:blue">commutative</em> if, additionally, the following formula is satisfied:
  $$\forall x: \forall y: x \circ y = y \circ x. $$

The notebook `FOL-Parser.ipynb` contains a notebook implementing a parser for first order logic.

In [ ]:
import { 
    LogicParser, 
    parse, 
    parseTerm,
    Term,     
    Variable,
    FuncTerm,
    Formula,          
    PredForm,      
    ConstForm,     
    NotForm,       
    BinaryForm,    
    QuantifierForm 
} from './FOL-Parser';

import { Tuple, RecursiveMap, RecursiveSet, Value } from 'recursive-set';

In [ ]:
type RS<T extends Value> = RecursiveSet<T>;
type RM<S extends Value, T extends Value> = RecursiveMap<S, T>;
const RS = RecursiveSet;
const RM = RecursiveMap;

We import a parser for FOL formulas.  This parser distinguishes between variables and function symbol as follows:
- A word starting with a lower case letter is interpreted as a *variable*.
- A word starting with an upper case letter is assumed to be a *function* or 
  *predicate symbol*.

Therefore, we represent the symbols from group theory as follows:
- The neutral element "$\mathrm{e}$" of group theory is represented as the nullary function symbol `E`.
- As our parser does not support using the symbol "$\circ$" as a binary operator, we will use the function symbol     
  `Multiply` to represent this operator.
- The predicate symbol "$=$" is repesented as `Equals`

Then the formulas of group theory can be represented as follows:

In [ ]:
const G1 = '∀x:Equals(Multiply(E(), x), x)';

In [ ]:
const G2 = '∀x:∃y:Equals(Multiply(y, x), E())';

In [ ]:
const G3 = '∀x:∀y:∀z:Equals(Multiply(Multiply(x, y), z), Multiply(x, Multiply(y, z)))';

In [ ]:
const G4 = '∀x:∀y:Equals(Multiply(x, y), Multiply(y, x))';

In [ ]:
const F1 = parse(G1);
`${F1}`

In [ ]:
const F2 = parse(G2);
`${F2}`

In [ ]:
const F3 = parse(G3);
`${F3}`

In [ ]:
const F4 = parse(G4);
`${F4}`

## A Structure for Group Theory

To evaluate formulas, we need a **Structure** $\mathcal{S} = \langle \mathcal{U}, \mathcal{J} \rangle$.

We implement this structure using `recursive-set` types to support tuple keys for functions and relations:
*   **RecursiveSet**: Defines the universe $\mathcal{U}$ and stores the true tuples for predicates.
*   **RecursiveMap**: Maps argument tuples to result values for functions.
*   **Tuple**: Used as the key for multi-argument lookups (e.g., `(x, y)`).

In [ ]:
type Universe                = RS<number>;
type FunctionInterpretation  = RM<Tuple<number[]>, number>;
type PredicateInterpretation = RS<Tuple<number[]>>;

interface Structure {
    universe:   Universe;
    functions:  RM<string, FunctionInterpretation>;
    predicates: RM<string, PredicateInterpretation>;
}

type VariableAssignment = RM<string, number>;

The smallest non-trivial group has just two elements.  Therefore, we can define the universe `U` as follows:

In [ ]:
const U: Universe = new RS(0, 1);

Next, we define the interpretation for the nullary function symbol `E`. We implement this as a `RecursiveMap` that maps the empty `Tuple()` to the element `0`.

In [ ]:
const NeutralElement: FunctionInterpretation = new RM();
NeutralElement.set(new Tuple(), 0);

The binary function symbol `Multiply` is implemented as the `RecursiveMap` named `Product`. It maps pairs of elements (represented as `Tuple(x, y)`) to their product.

In [ ]:
const Product: FunctionInterpretation = new RM();
Product.set(new Tuple(0, 0), 0);
Product.set(new Tuple(0, 1), 1);
Product.set(new Tuple(1, 0), 1);
Product.set(new Tuple(1, 1), 0);

The predicate symbol `Equals` is implemented as the binary relation `Identity`. We define it as a `RecursiveSet` containing all pairs `Tuple(x, x)` for every element $x$ in the universe.

In [ ]:
const Identity: PredicateInterpretation = U.map(x => new Tuple(x, x));

Now, the interpretation $\mathcal{J}$ consists of two maps:
1.  `functions`: Maps function names (strings) to their `FunctionInterpretation` (which are `RecursiveMaps`).
2.  `predicates`: Maps predicate names (strings) to their `PredicateInterpretation` (which are `RecursiveSets`).

In [ ]:
const functions = new RM<string, FunctionInterpretation>();
functions.set('E',        NeutralElement);
functions.set('Multiply', Product);

const predicates = new RM<string, PredicateInterpretation>();
predicates.set('Equals', Identity);

We define the first order structure $\mathcal{S}$. Mathematically, this is a pair $(\mathcal{U}, \mathcal{J})$. In our code, we represent it as an object containing the universe $\mathcal{U}$ along with the function and predicate interpretations that make up $\mathcal{J}$.

In [ ]:
const S: Structure = { 
    universe:   U, 
    functions:  functions, 
    predicates: predicates 
};

Finally, we define the *variable assignment* $\mathcal{I}$ for the variables $x$, $y$, and $z$. 

In [ ]:
const I: VariableAssignment = new RM<string, number>();
I.set('x', 0);
I.set('y', 1);
I.set('z', 0);
I;

The function `evalTerm(t, S, I)` evaluates a term $t$ into an element of the universe.
*   If $t$ is a **variable**, we look up its value in the variable assignment $\mathcal{I}$.
*   If $t$ is a **function application** $f(t_1, \dots, t_n)$, we recursively evaluate all arguments and then look up the result in the interpretation of $f$ given by $\mathcal{S}$.

In [ ]:
function evalTerm(t: Term, S: Structure, I: VariableAssignment): number {
    if (typeof t == 'string') {
        return I.get(t);
    } 
    const argValues: number[] = t.args.map(arg => evalTerm(arg, S, I));
    const fJ = S.functions.get(t.symbol);
    return fJ.get(new Tuple(...argValues));
}

In [ ]:
const t = parseTerm('Multiply(E(), x)');
`${t}`

In [ ]:
evalTerm(t, S, I);

The function `evalAtomic(a, S, I)` evaluates an atomic formula (a predicate application).
It recursively evaluates the predicate's arguments to a tuple of values and then checks if this tuple is contained in the predicate's interpretation (relation) within the structure $\mathcal{S}$.

In [ ]:
function evalAtomic(
    F: PredForm, 
    S: Structure, 
    I: VariableAssignment
): boolean {
    const argValues = Array.from(F.args, arg => evalTerm(arg, S, I));
    return S.predicates.get(F.symbol).has(new Tuple(...argValues));
}

In [ ]:
const f = parse('Equals(Multiply(E(), x), x)') as PredForm;
`${f}`

In [ ]:
evalAtomic(f, S, I);

Given a variable assignment $\mathcal{I}$, a variable $x$, and an element $c$ from the universe $\mathcal{U}$, the function $\texttt{modify}(\mathcal{I}, x, c)$ computes the variable assignment $\mathcal{I}[x/c]$ which is defined for all variables $y$ as follows:
$$ 
I[x/c](y) = \left\{ \begin{array}{ll}
                        c     & \mbox{if $x = y$,}  \\
                        I(y)  & \mbox{otherwise.}
                        \end{array}
               \right.
$$

In [ ]:
function modify(I: VariableAssignment, variable: string, value: number): VariableAssignment {
    const Is = I.mutableCopy();
    Is.set(variable, value);
    return Is;
}

The function `evalFormula(F, S, I)` computes the truth value (`true` or `false`) of a formula $F$.
It recursively evaluates the components of the formula:
*   **Connectives** ($\land, \lor, \dots$) are mapped to their TypeScript equivalents (`&&`, `||`, etc.).
*   **Quantifiers** ($\forall x, \exists x$) iterate over all elements of the universe $\mathcal{U}$. We use `modify(I, x, c)` to update the variable assignment for the current element $c$.

In [ ]:
false <= true

In [ ]:
function evalFormula(F: Formula, S: Structure, I: VariableAssignment): boolean {
    if (F instanceof PredForm) {
        return evalAtomic(F, S, I);
    }
    if (F instanceof ConstForm) {
        return F.value == '⊤';
    }
    if (F instanceof NotForm) {
        return !evalFormula(F.body, S, I);
    }
    if (F instanceof BinaryForm) {
        const G = F.left;
        const H = F.right;   
        switch (F.op) {
            case '∧':  return  evalFormula(G, S, I) && evalFormula(H, S, I);
            case '∨':  return  evalFormula(G, S, I) || evalFormula(H, S, I);
            case '→': return  evalFormula(G, S, I) <= evalFormula(H, S, I);
            case '↔': return  evalFormula(G, S, I) == evalFormula(H, S, I);
        }
    }
    if (F instanceof QuantifierForm) {
        const x = F.variable;
        const G = F.body;       
        if (F.op == '∀') {
            return S.universe.every(c => evalFormula(G, S, modify(I, x, c)));
        } 
        if (F.op == '∃') {
            return S.universe.some(c => evalFormula(G, S, modify(I, x, c)));
        }
    }
}

## Checking whether $\mathcal{S}$ is a Group

In [ ]:
console.log(`evalFormula(${G1}, S, I) = ${evalFormula(F1, S, I)}`);
console.log(`evalFormula(${G2}, S, I) = ${evalFormula(F2, S, I)}`);
console.log(`evalFormula(${G3}, S, I) = ${evalFormula(F3, S, I)}`);
console.log(`evalFormula(${G4}, S, I) = ${evalFormula(F4, S, I)}`);

This shows that the structure $\mathcal{S}$ defined above is indeed a group.  Furthermore, it is a *commutative* group.

## Another Example

Let's show that the formula $\forall x: \exists y:p(x,y) \rightarrow \exists y:\forall x:p(x,y)$ is not *universally valid*, i.e. let's show the following:
$$ \not\models \forall x: \exists y:p(x,y) \rightarrow \exists y:\forall x:p(x,y) $$

In [ ]:
const G = '∀x:∃y:P(x,y)→∃y:∀x:P(x,y)';

Our aim is to construct a structure $\mathcal{S} = \langle\mathcal{U}, \mathcal{J} \rangle$  such that 
$$
\mathcal{S}(F) = \mathtt{false}.
$$ 

In [ ]:
const U: Universe = new RecursiveSet(0, 1);

In [ ]:
const pJ: PredicateInterpretation = new RecursiveSet(new Tuple(0, 0), new Tuple(1, 1));
const predicates = new RecursiveMap<string, PredicateInterpretation>();
predicates.set('P', pJ);
const functions = new RecursiveMap<string, FunctionInterpretation>();

In [ ]:
const S: Structure = { 
    universe: U, 
    functions: functions, 
    predicates: predicates 
};

In [ ]:
const I: VariableAssignment = new RecursiveMap<string, number>();
I.set('x', 0);
I.set('y', 0);

In [ ]:
const F = parse(G);
`${F}`

In [ ]:
evalFormula(F, S, I);